In [ ]:
from imitation.scripts.generate_ranked_trajectories import generate_policies_and_videos, generate_trajectories, record_trajectory_video 

ENV_NAME = "CartPole-v1"
ITERATION_STEP = 1000
NUM_POLICIES = 10

video_entries, policy_entries = generate_policies_and_videos(NUM_POLICIES, ITERATION_STEP, ENV_NAME)
trajectory_entries = generate_trajectories(policy_entries, ENV_NAME)

for i in range(0, len(trajectory_entries)):
    print(len(trajectory_entries[i]["trajectory"].obs)) # why is reward capped at 500 ? 


In [ ]:
# Building ranked pairs as a learning dataset
pairs = []
for i, t1 in enumerate(trajectory_entries):
    for j, t2 in enumerate(trajectory_entries):
        if i >= j: continue
        y = 1.0 if sum(t1["trajectory"].rews) > sum(t2["trajectory"].rews) else 0.0
        pairs.append((t1["trajectory"].obs, t2["trajectory"].obs, y))

In [ ]:
#TODO: add collatefn to add padding (or do it ealier) 

from torch.utils.data import Dataset, DataLoader
import torch

class PrefPairsDS(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        o1, o2, y = self.pairs[i]               # (T, obs_dim), (T, obs_dim), scalar
        return torch.from_numpy(o1), torch.from_numpy(o2), torch.tensor(y, dtype=torch.float32)


train_loader = DataLoader(PrefPairsDS(pairs), batch_size=64, shuffle=True)

In [ ]:
# TODO: correct and test this code (is this the correct loss function ??)
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

obs_dim = gym.make(ENV_NAME).observation_space.shape[0]

class RewardMLP(nn.Module):
    def __init__(self, obs_dim, hid=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hid), nn.Tanh(),
            nn.Linear(hid, hid), nn.Tanh(),
            nn.Linear(hid, 1)
        )
    def forward(self, x):            # x: (B, D)
        return self.net(x).squeeze(-1)

reward_net = RewardMLP(obs_dim)
opt = torch.optim.Adam(reward_net.parameters(), lr=3e-4)
device = torch.device("cpu"); reward_net.to(device)

def pred_frag_return(obs_seq):       # (B, T, D) -> (B,)
    B,T,D = obs_seq.shape
    r = reward_net(obs_seq.reshape(B*T, D)).view(B,T).sum(dim=1)
    return r

def bt_loss(r1, r2, y):              # y in {0,1}
    return F.binary_cross_entropy_with_logits(r1 - r2, y)



In [ ]:
# TODO: correct and test the training cycle, add visualization

for step, (o1, o2, y) in enumerate(train_loader):
    o1, o2, y = o1.to(device), o2.to(device), y.to(device)
    T = min(o1.shape[1], o2.shape[1]); o1, o2 = o1[:,:T], o2[:,:T]
    r1, r2 = pred_frag_return(o1), pred_frag_return(o2)
    loss = bt_loss(r1, r2, y)
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(reward_net.parameters(), 5.0)
    opt.step()
    if step % 200 == 0: print(f"{step:05d}  loss={loss.item():.4f}")
    if step >= 2000: break